**โจทย์:** ประเมินความเข้มกลิ่นจากค่าของเซนเซอร์ เพื่อตรวจจับเหตุการณ์ผิดปกติ (Anomaly Detection)  
**กระบวนการ:** Data Science Process 6 ขั้นตอน

---
## Step 1 — Setting the Research Goal

### บริบทของปัญหา
หน้างาน (โรงงาน/โรงบำบัดน้ำเสีย) ติดตั้งสถานีตรวจวัดกลิ่นต่อเนื่อง ชุมชนรอบข้างร้องเรียนเรื่องกลิ่นรบกวนบ่อยขึ้น ผู้จัดการต้องการเครื่องมือที่ช่วยให้:
- เข้าใจสถานการณ์กลิ่นจากข้อมูล sensor
- ตอบข้อร้องเรียนได้อย่างมีหลักฐาน
- ตัดสินใจได้เร็วขึ้นว่าควรแก้ไขที่จุดใด เมื่อใด

### Research Goal
> **"ประเมินความเข้มกลิ่น (D/T) จากค่าของ sensor ทั้ง 8 ตัว และตรวจจับช่วงเวลาที่กลิ่นผิดปกติ เพื่อสนับสนุนการตัดสินใจของหน้างาน"**

---
## Step 2 — Retrieving Data

ข้อมูลมาจากสถานีตรวจวัดกลิ่น ENose บันทึกค่าทุก **1 นาที** ต่อเนื่องประมาณ **91 วัน**  
ไฟล์: `Export.csv` — เป็นข้อมูลจริงที่ยังไม่ผ่านการทำความสะอาด

### หมายเหตุการโหลดไฟล์
ไฟล์นี้มีบรรทัดแรกเป็น `sep=,` ซึ่งเป็น Excel dialect — ต้องใช้ `skiprows=1` เพื่อข้ามบรรทัดนั้น

In [1]:
import pandas as pd
import numpy as np

# โหลดข้อมูล — skiprows=1 เพื่อข้ามบรรทัด sep=,
df = pd.read_csv("Export.csv", skiprows=1)

# แปลง Time เป็น datetime
df["Time"] = pd.to_datetime(df["Time"])

print(f"โหลดสำเร็จ: {df.shape[0]:,} แถว, {df.shape[1]} คอลัมน์")
print(f"ช่วงเวลา: {df['Time'].min()} ถึง {df['Time'].max()}")

โหลดสำเร็จ: 130,343 แถว, 17 คอลัมน์
ช่วงเวลา: 2026-02-12 00:00:00 ถึง 2026-05-14 12:31:00


### 2.1 ดูโครงสร้างข้อมูล

In [2]:
# ดูตัวอย่างข้อมูล
df.head(5)

,Time,D/T,Wind Direction,Wind Speed,Temperature,Relative Humidity,PM 2.5,Atmospheric Pressure,Sensor 1,Sensor 2,Sensor 3,Sensor 4,Sensor 5,Sensor 6,Sensor 7,Sensor 8,Smell Prediction
0,2026-02-12 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
1,2026-02-12 00:01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
2,2026-02-12 00:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
3,2026-02-12 00:03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
4,2026-02-12 00:04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient


In [3]:
# ดู data type ของแต่ละคอลัมน์
df.dtypes

Time                    datetime64[ns]
D/T                            float64
Wind Direction                 float64
Wind Speed                     float64
Temperature                    float64
Relative Humidity              float64
PM 2.5                         float64
Atmospheric Pressure           float64
Sensor 1                       float64
Sensor 2                       float64
Sensor 3                       float64
Sensor 4                       float64
Sensor 5                       float64
Sensor 6                       float64
Sensor 7                       float64
Sensor 8                       float64
Smell Prediction                object
dtype: object

### 2.2 ตรวจสอบ Missing Values

In [2]:
# นับ missing values แต่ละคอลัมน์
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})
missing_summary[missing_summary["Missing Count"] > 0]

,Missing Count,Missing %
D/T,421,0.32
Wind Direction,422,0.32
Wind Speed,422,0.32
Temperature,422,0.32
Relative Humidity,422,0.32
PM 2.5,422,0.32
Atmospheric Pressure,422,0.32
Sensor 1,421,0.32
Sensor 2,421,0.32
Sensor 3,421,0.32


### 2.3 ดู Smell Prediction — target variable

In [3]:
# นับค่าใน Smell Prediction (รวม NaN)
print("=== Smell Prediction value_counts ===")
print(df["Smell Prediction"].value_counts(dropna=False))

print("\n=== แถวที่ Sensor ว่างทุกตัว ===")
sensor_cols = [f"Sensor {i}" for i in range(1, 9)]
all_sensor_null = df[sensor_cols].isnull().all(axis=1)
print(f"Sensor ว่างทุกตัว : {all_sensor_null.sum():,} แถว")
print(f"Sensor มีข้อมูล   : {(~all_sensor_null).sum():,} แถว")

=== Smell Prediction value_counts ===
Smell Prediction
Ambient        85504
NaN            44838
hackathon#2        1
Name: count, dtype: int64

=== แถวที่ Sensor ว่างทุกตัว ===
Sensor ว่างทุกตัว : 421 แถว
Sensor มีข้อมูล   : 129,922 แถว


### 2.4 สรุปสิ่งที่พบ

| ประเด็น | รายละเอียด | แนวทาง Step 3 |
|---|---|---|
| Sensor NaN 421 แถว | ~0.3% — sensor ว่างทุกตัว | ตัดทิ้ง |
| `hackathon#2` 1 แถว | ไม่ใช่ข้อมูลจริง | ตัดทิ้ง |
| Smell Prediction NaN 44,838 แถว | 34% — ยังไม่ทราบสาเหตุ | ต้องวิเคราะห์เพิ่ม |
| มีแค่ "Ambient" | ทั้งที่สเปกรองรับ 5 โปรไฟล์ | ต้องใช้ D/T แทน label |